# 12 · Typical-setting context: co-occurrence control and LVIS

Follows notebook 11, which showed that BV domain clustering survives when compared categories come from different
subjects (shared *scenes* do not explain it). Two remaining questions from review:

- **B · Typical settings.** Food is near tables and plates in every home, so context can recur across homes.
  Does "same CDI domain" still predict embedding distance once category co-occurrence is controlled, and does BV
  still cluster more than THINGS / VEDB under the same control?
- **D · Is context sufficient?** LVIS box crops come from third-person scene photos and keep their backgrounds.
  If having context in the crop were enough, LVIS should cluster like BV.

**Decisions**
- Co-occurrence is counted over **frames** (1 fps) from `crops_index.csv.gz`, using the analysis crop set:
  CLIP image–label similarity ≥ 0.27, valid129 categories, BV subject 00270001 excluded. A check cell confirms the
  BV crop count matches the manuscript Stage 0 total (3,816,220).
- Two covariates per category pair: **direct co-occurrence** (Jaccard over frames) and **shared-context similarity**
  (cosine between the two categories' PPMI co-occurrence profiles over all 129 categories, leaving out the pair's
  own two columns). The second is the typical-setting proxy: two foods may rarely share a frame but share tables.
- Model on category pairs (upper triangle), every variable rank-transformed and standardized:
  distance ~ same-domain [+ co-occurrence + shared context]. The **same-domain coefficient** (sign flipped so
  larger = tighter clusters) is comparable across datasets because ranks remove RDM scale (it tracks AUC).
- Inference on the controlled same-domain coefficient: domain-label permutation (count-preserving) with the
  covariates held fixed. Dataset differences: **within-domain category bootstrap**, self-pairs dropped, the same
  draws for every dataset. Every dataset is controlled with the **same covariates** (BV co-occurrence unless noted),
  so differences compare like with like.
- Embeddings: Stage 0 raw category means, z-scored within each comparison set. BV is **2025.2** (it matches the
  crop index used for co-occurrence), not the 868-hour tables used by notebooks 05–09.
- Co-occurrence also tracks meaning (things that go together are related), so controlling it is conservative:
  it can remove genuine semantic structure along with context. Read the *controlled differences* between datasets,
  not the controlled coefficients alone.

**Outputs:** `outputs/context_controls/results/` (tables, including the supplementary tables) and `cache/`
(co-occurrence matrices; small, kept in git ignore with notebook 11's cache).

In [ ]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

# Manuscript figure style (same as analysis/qwen-open-vocab/fig_style.py): Lato, bold axis labels and legends.
import matplotlib.axes, matplotlib.axis, matplotlib.figure
from matplotlib import font_manager
for _f in (Path.cwd().resolve().parents[1] / "talk-figures" / "unr-2026" / "fonts").glob("*.ttf"):
    font_manager.fontManager.addfont(str(_f))
plt.rcParams.update({
    "font.family": ["Lato", "DejaVu Sans", "sans-serif"], "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.labelweight": "bold", "figure.titlesize": 12, "figure.titleweight": "bold",
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#444444", "axes.labelcolor": "#222222",
    "text.color": "#222222", "xtick.color": "#444444", "ytick.color": "#444444"})
def _bold_legend(orig):
    if getattr(orig, "_bold", False):
        return orig
    def legend(self, *args, **kw):
        prop = dict(kw.pop("prop", {}) or {}); prop.setdefault("weight", "bold")
        prop.setdefault("size", kw.pop("fontsize", plt.rcParams["legend.fontsize"]))
        return orig(self, *args, prop=prop, **kw)
    legend._bold = True
    return legend
matplotlib.axes.Axes.legend = _bold_legend(matplotlib.axes.Axes.legend)
matplotlib.figure.Figure.legend = _bold_legend(matplotlib.figure.Figure.legend)
_tick_init = matplotlib.axis.Tick.__init__
if not getattr(_tick_init, "_bold", False):
    def _bold_tick(self, *a, **k):                       # bold tick labels (no rcParam for tick label weight)
        _tick_init(self, *a, **k); self.label1.set_fontweight("bold"); self.label2.set_fontweight("bold")
    _bold_tick._bold = True; matplotlib.axis.Tick.__init__ = _bold_tick
    _grid = matplotlib.axes.Axes.grid                    # no grid lines in any figure
    matplotlib.axes.Axes.grid = lambda self, visible=None, which="major", axis="both", **kw: _grid(self, False, which=which, axis=axis)
plt.rcParams["axes.grid"] = False
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [ ]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

# Inputs for this notebook (read only).
BV_CROPS_INDEX = Path("/ccn2a/dataset/babyview/2025.2/outputs/image_embeddings/babyview_full/crops_index.csv.gz")
VEDB_CROPS_INDEX = Path("/ccn2a/dataset/babyview/related_data/vedb_1612/outputs/image_embeddings/vedb_full_adults/crops_index.csv.gz")
STAGE0 = MANUSCRIPT_DIR / "exemplar_set_embeddings"
RAW = {
    "bv": str(STAGE0 / "2025.2" / "valid129" / "bv_{m}_exemplar_avg_raw_within_valid129.csv"),
    "things": str(STAGE0 / "valid129" / "things_{m}_exemplar_avg_raw_within_valid129.csv"),
    "vedb": str(VEDB_VALID129 / "bv_{m}_exemplar_avg_raw_within_valid129.csv"),
    "lvis": str(EXEMPLAR_SETS / "lvis_exclusive104" / "lvis_{m}_exemplar_avg_raw_within_lvis_exclusive104.csv"),
}
LVIS_CATEGORIES = EXEMPLAR_SETS / "lvis_exclusive104" / "categories.txt"
CONTEXT_SPLIT_RESULTS = OUT / "context_split" / "results"      # notebook 11 (for Supplementary Table Y)

CC_OUT = OUT / "context_controls"
CC_CACHE, CC_RESULTS = CC_OUT / "cache", CC_OUT / "results"
for d in (CC_CACHE, CC_RESULTS):
    d.mkdir(parents=True, exist_ok=True)

## Parameters

In [ ]:
MIN_SIM = 0.27
EXCLUDE_BV_SUBJECT_PREFIX = "S00270001"
MIN_VEDB_FRAMES = 20     # VEDB categories need this many frames for a co-occurrence profile
N_PERM = 2000            # domain-label permutations (controlled same-domain coefficient)
N_BOOT = 1000            # category bootstrap draws (dataset differences)
RECOMPUTE_CACHE = False
LABEL = {"bv": "BabyView", "vedb": "VEDB", "things": "THINGS", "lvis": "LVIS"}

## Helpers

In [ ]:
def load_cdi_semantic_map() -> dict[str, str]:
    """``uni_lemma`` → CDI semantic domain from ``data/cdi_words.csv`` (same as notebook 09)."""
    df = pd.read_csv(DATA_DIR / "cdi_words.csv", usecols=["uni_lemma", "category"])
    df["uni_lemma"] = df["uni_lemma"].astype(str).str.strip().str.lower()
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.drop_duplicates(subset=["uni_lemma"], keep="first").set_index("uni_lemma")["category"].to_dict()


CATS = sorted(l.strip().lower() for l in VALID129_TXT.read_text().splitlines() if l.strip())
CDI = load_cdi_semantic_map()


def load_raw(ds: str, model: str, cats: list[str]) -> np.ndarray:
    df = pd.read_csv(RAW[ds].format(m=model)); df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.set_index("category").loc[cats].filter(like="dim_").to_numpy(np.float64)


def zscore_rows(X: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    return (X - X.mean(axis=0)) / (X.std(axis=0) + eps)


def rdm(X: np.ndarray) -> np.ndarray:
    X = X / np.linalg.norm(X, axis=1, keepdims=True); return 1 - X @ X.T


def rank_z(v: np.ndarray) -> np.ndarray:
    r = rankdata(v); return (r - r.mean()) / r.std()


def ols(y: np.ndarray, X: list[np.ndarray]):
    A = np.column_stack([np.ones(len(y))] + X); b, *_ = np.linalg.lstsq(A, y, rcond=None)
    return b[1:], 1 - (y - A @ b).var() / y.var()


def domain_bootstrap(sem: np.ndarray, n: int, seed: int) -> list[np.ndarray]:
    rng = np.random.default_rng(seed); doms = sorted(set(sem))
    return [np.concatenate([rng.choice(np.flatnonzero(sem == d), (sem == d).sum()) for d in doms]) for _ in range(n)]


def ci_p(x: np.ndarray):
    lo, hi = np.percentile(x, [2.5, 97.5]); return lo, hi, float(2 * min((x <= 0).mean(), (x >= 0).mean()))

## Stage A · frame-level co-occurrence

`C[i, j]` = number of frames containing both categories (diagonal = frames containing the category).

In [ ]:
import scipy.sparse as sp


def build_cooccurrence(index_path: Path, out: Path) -> None:
    cix = {c: i for i, c in enumerate(CATS)}; parts = []
    for ch in pd.read_csv(index_path, usecols=["video_id", "frame_number", "class_name", "clip_sim"], chunksize=2_000_000):
        ch = ch[ch["clip_sim"] >= MIN_SIM].copy()
        ch["class_name"] = ch["class_name"].astype(str).str.strip().str.lower()
        ch = ch[ch["class_name"].isin(cix) & ~ch["video_id"].astype(str).str.startswith(EXCLUDE_BV_SUBJECT_PREFIX)]
        parts.append(ch[["video_id", "frame_number", "class_name"]])
    df = pd.concat(parts, ignore_index=True)
    frame = pd.factorize(df["video_id"].astype(str) + "#" + df["frame_number"].astype(str))[0]
    B = sp.csr_matrix((np.ones(len(df), np.float32), (frame, df["class_name"].map(cix).to_numpy())),
                      shape=(frame.max() + 1, len(CATS)))
    B.sum_duplicates(); B.data[:] = 1
    np.savez(out, cats=np.array(CATS), C=(B.T @ B).toarray(), n_frames=B.shape[0], n_crops=len(df))


COOC_PATH = {"bv": CC_CACHE / "cooccurrence_bv_2025_2.npz", "vedb": CC_CACHE / "cooccurrence_vedb.npz"}
for ds, idx_path in (("bv", BV_CROPS_INDEX), ("vedb", VEDB_CROPS_INDEX)):
    if RECOMPUTE_CACHE or not COOC_PATH[ds].exists():
        build_cooccurrence(idx_path, COOC_PATH[ds])
COOC = {ds: np.load(p) for ds, p in COOC_PATH.items()}
for ds, z in COOC.items():
    assert list(z["cats"]) == CATS
    print(f"{ds:5s} crops {int(z['n_crops']):,}  frames {int(z['n_frames']):,}")
assert int(COOC["bv"]["n_crops"]) == 3_816_220, "BV crop set should match manuscript Stage 0 (2025.2)"

## Pair covariates

In [ ]:
def pair_covariates(z, cats: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """Jaccard co-occurrence and shared-context similarity (PPMI profile cosine, the pair's own columns left out)."""
    idx = np.array([CATS.index(c) for c in cats])
    Cf = z["C"].astype(float); nf = np.diag(Cf).copy(); N = float(z["n_frames"])
    C = Cf[np.ix_(idx, idx)]; n = nf[idx]
    jac = C / (n[:, None] + n[None, :] - C + 1e-12)
    ppmi = np.maximum(np.log((Cf * N + 1e-12) / (np.outer(nf, nf) + 1e-12)), 0); np.fill_diagonal(ppmi, 0)
    P = ppmi[idx]; prof = np.zeros((len(idx), len(idx)))
    for a, b in combinations(range(len(idx)), 2):
        keep = np.ones(P.shape[1], bool); keep[[idx[a], idx[b]]] = False; u, v = P[a, keep], P[b, keep]
        prof[a, b] = prof[b, a] = u @ v / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12)
    return jac, prof


def pair_design(R, sem, jac, prof, sel=None):
    """Rank-z response and predictors over category pairs (bootstrap-safe: self-pairs dropped)."""
    sel = np.arange(len(sem)) if sel is None else sel
    s = np.asarray(sem)[sel]; iu = np.triu_indices(len(sel), 1); keep = sel[iu[0]] != sel[iu[1]]
    sub = lambda M: M[np.ix_(sel, sel)][iu][keep]
    same = (s[iu[0]] == s[iu[1]])[keep].astype(float)
    return rank_z(sub(R)), (same - same.mean()) / same.std(), rank_z(sub(jac)), rank_z(sub(prof)), same


def domain_coefficients(R, sem, jac, prof, sel=None) -> tuple[float, float]:
    """Same-domain coefficient alone and controlled; sign flipped so larger = tighter domain clusters."""
    y, xs, xj, xp, _ = pair_design(R, sem, jac, prof, sel)
    return -ols(y, [xs])[0][0], -ols(y, [xs, xj, xp])[0][0]

## B · Does the domain effect survive co-occurrence controls?

Comparison sets: `bv_things` (129 categories) and `bv_vedb_things` (VEDB categories with ≥ `MIN_VEDB_FRAMES`
frames). Each dataset is tested with BV co-occurrence and, in the three-way set, also with VEDB's.

In [ ]:
def comparison_sets() -> dict[str, tuple[list[str], list[str]]]:
    vedb_cats = set(pd.read_csv(RAW["vedb"].format(m="clip"), usecols=["category"])["category"].str.strip().str.lower())
    n_vedb = np.diag(COOC["vedb"]["C"])
    three = [c for c in CATS if c in vedb_cats and n_vedb[CATS.index(c)] >= MIN_VEDB_FRAMES]
    lvis = [l.strip().lower() for l in LVIS_CATEGORIES.read_text().splitlines() if l.strip()]
    return {"bv_things": (CATS, ["bv", "things"]), "bv_vedb_things": (three, ["bv", "vedb", "things"]),
            "bv_vedb_things_lvis": (lvis, ["bv", "vedb", "things", "lvis"])}


SETS = comparison_sets()
COV = {(name, src): pair_covariates(COOC[src], cats) for name, (cats, _) in SETS.items() for src in ("bv", "vedb")}
RDMS = {(name, m, ds): rdm(zscore_rows(load_raw(ds, m, cats))) for name, (cats, dss) in SETS.items()
        for m in MODELS for ds in dss}
print({name: len(c) for name, (c, _) in SETS.items()})

rows = []
for name in ("bv_things", "bv_vedb_things"):
    cats, dss = SETS[name]; sem = np.array([CDI.get(c, "other") for c in cats])
    for m in MODELS:
        for ds in dss:
            for src in (["bv", "vedb"] if "vedb" in dss else ["bv"]):
                jac, prof = COV[(name, src)]
                y, xs, xj, xp, same = pair_design(RDMS[(name, m, ds)], sem, jac, prof)
                b0, r2_dom = ols(y, [xs]); b1, r2_full = ols(y, [xs, xj, xp]); _, r2_cov = ols(y, [xj, xp])
                rng = np.random.default_rng(5); iu = np.triu_indices(len(sem), 1); null = np.empty(N_PERM)
                for k in range(N_PERM):
                    s = rng.permutation(sem); sm = (s[iu[0]] == s[iu[1]]).astype(float)
                    null[k] = ols(y, [(sm - sm.mean()) / sm.std(), xj, xp])[0][0]
                rows.append({"comparison": name, "n_categories": len(cats), "model": m, "dataset": ds, "cooccurrence_from": src,
                             "domain_coef_alone": -b0[0], "domain_coef_controlled": -b1[0], "share_retained": b1[0] / b0[0],
                             "p_perm_controlled": (1 + (null <= b1[0]).sum()) / (N_PERM + 1),
                             "cooccurrence_coef": -b1[1], "shared_context_coef": -b1[2],
                             "r2_domain_alone": r2_dom, "r2_covariates_alone": r2_cov, "r2_full": r2_full,
                             "unique_r2_domain": r2_full - r2_cov})
coef = pd.DataFrame(rows)
coef.to_csv(CC_RESULTS / "cooccurrence_control_coefficients.csv", index=False)
display(coef.drop(columns=["r2_domain_alone", "r2_covariates_alone"]).round(3))

### Dataset differences before and after the same controls (BV co-occurrence for every dataset)

Positive = first dataset clusters more by domain.

In [ ]:
diff_rows = []
for name in ("bv_things", "bv_vedb_things"):
    cats, dss = SETS[name]; sem = np.array([CDI.get(c, "other") for c in cats]); jac, prof = COV[(name, "bv")]
    draws = domain_bootstrap(sem, N_BOOT, 3)
    for m in MODELS:
        obs = {ds: domain_coefficients(RDMS[(name, m, ds)], sem, jac, prof) for ds in dss}
        bt = {ds: np.array([domain_coefficients(RDMS[(name, m, ds)], sem, jac, prof, d) for d in draws]) for ds in dss}
        for a, b in combinations(dss, 2):
            for k, lab in enumerate(("alone", "controlled")):
                lo, hi, p = ci_p(bt[a][:, k] - bt[b][:, k])
                diff_rows.append({"comparison": name, "n_categories": len(cats), "model": m, "pair": f"{a}_minus_{b}",
                                  "domain_coef": lab, "diff": obs[a][k] - obs[b][k], "ci_lo": lo, "ci_hi": hi, "p_boot": p})
coef_diff = pd.DataFrame(diff_rows)
coef_diff.to_csv(CC_RESULTS / "cooccurrence_control_dataset_differences.csv", index=False)
display(coef_diff.pivot_table(index=["comparison", "model", "pair"], columns="domain_coef", values=["diff", "p_boot"]).round(3))

## D · LVIS: in-scene third-person crops (104 LVIS-exclusive categories)

Every dataset z-scored within the 104 categories. AUC = P(between-domain distance > within-domain distance);
raw Δ and Δ/SD as in notebook 09; same-domain coefficient alone and with BV co-occurrence controls.
Caveat: LVIS uses ground-truth boxes with no CLIP or size filter (tiny and occluded objects included).

In [ ]:
def set_metrics(R, sem, jac, prof, sel=None) -> dict:
    sel = np.arange(len(sem)) if sel is None else sel
    s = np.asarray(sem)[sel]; iu = np.triu_indices(len(sel), 1); keep = sel[iu[0]] != sel[iu[1]]
    v = R[np.ix_(sel, sel)][iu][keep]; same = (s[iu[0]] == s[iu[1]])[keep]; w, b = v[same], v[~same]
    rk = rankdata(np.concatenate([w, b])); auc = (rk[len(w):].sum() - len(b) * (len(b) + 1) / 2) / (len(w) * len(b))
    alone, controlled = domain_coefficients(R, sem, jac, prof, sel)
    return {"auc": auc, "delta": b.mean() - w.mean(), "delta_sd": (b.mean() - w.mean()) / v.std(),
            "domain_coef_alone": alone, "domain_coef_controlled": controlled}


name = "bv_vedb_things_lvis"; cats, dss = SETS[name]; sem = np.array([CDI.get(c, "other") for c in cats])
jac, prof = COV[(name, "bv")]; draws = domain_bootstrap(sem, N_BOOT, 3)
lv_rows, lv_diff = [], []
for m in MODELS:
    obs = {ds: set_metrics(RDMS[(name, m, ds)], sem, jac, prof) for ds in dss}
    bt = {ds: pd.DataFrame([set_metrics(RDMS[(name, m, ds)], sem, jac, prof, d) for d in draws]) for ds in dss}
    for ds in dss:
        lv_rows.append({"model": m, "dataset": ds, "n_categories": len(cats), **obs[ds],
                        **{f"{k}_ci_lo": np.percentile(bt[ds][k], 2.5) for k in obs[ds]},
                        **{f"{k}_ci_hi": np.percentile(bt[ds][k], 97.5) for k in obs[ds]}})
    for a, b in (("bv", "lvis"), ("vedb", "lvis"), ("things", "lvis"), ("bv", "things"), ("bv", "vedb")):
        for k in obs[a]:
            lo, hi, p = ci_p(bt[a][k].to_numpy() - bt[b][k].to_numpy())
            lv_diff.append({"model": m, "pair": f"{a}_minus_{b}", "metric": k, "diff": obs[a][k] - obs[b][k],
                            "ci_lo": lo, "ci_hi": hi, "p_boot": p})
lvis_summary, lvis_diff = pd.DataFrame(lv_rows), pd.DataFrame(lv_diff)
lvis_summary.to_csv(CC_RESULTS / "lvis_comparison_summary.csv", index=False)
lvis_diff.to_csv(CC_RESULTS / "lvis_comparison_dataset_differences.csv", index=False)
print("domains:", pd.Series(sem).value_counts().to_dict())
display(lvis_summary[["model", "dataset", "auc", "delta", "delta_sd", "domain_coef_alone", "domain_coef_controlled"]].round(3))
display(lvis_diff.pivot_table(index=["model", "pair"], columns="metric", values="p_boot").round(3))

## Supplementary tables (manuscript)

- **Table Y** · subject split (notebook 11): dataset differences on the split-averaged cross-half RDM.
- **Table Z** · co-occurrence control (B) and LVIS (D).

Written as CSV (numbers) and Markdown (formatted, for the supplement draft).

In [ ]:
def fmt(d, lo, hi, p, nd=3):
    ps = "p < .001" if p < 0.001 else "p = " + f"{p:.3f}".lstrip("0")
    return f"{d:+.{nd}f} [{lo:+.{nd}f}, {hi:+.{nd}f}], {ps}"


def to_markdown(df: pd.DataFrame) -> str:
    """Pipe table without the optional tabulate dependency."""
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


PAIR_LABEL = lambda p: " − ".join(LABEL[x] for x in p.split("_minus_"))
MODEL_LABEL = {"clip": "CLIP", "dinov3": "DINOv3"}

# Table Y
cs = pd.read_csv(CONTEXT_SPLIT_RESULTS / "context_split_dataset_differences.csv")
cs = cs[cs.comparison == "bv_vedb_things"]
ty = (cs.assign(cell=[fmt(r.diff, r.ci_lo, r.ci_hi, r.p_boot_two_sided) for r in cs.itertuples()],
                Model=cs.model.map(MODEL_LABEL), Contrast=cs.pair.map(PAIR_LABEL))
        .pivot_table(index=["Model", "Contrast"], columns="metric", values="cell", aggfunc="first")
        .rename(columns={"auc": "AUC", "delta_sd": "Δ / SD", "delta": "Raw Δ (cosine)"})[["AUC", "Δ / SD", "Raw Δ (cosine)"]])
ty.to_csv(CC_RESULTS / "supp_table_Y_subject_split_differences.csv")

# Table Z: rows = contrast; columns = domain coefficient alone / controlled (B: 129 and 115 sets), LVIS set AUC
zb = coef_diff.assign(cell=[fmt(r.diff, r.ci_lo, r.ci_hi, r.p_boot) for r in coef_diff.itertuples()])
zb = zb[(zb.comparison == "bv_things") | (zb.pair != "bv_minus_things")]   # BV − THINGS from the full 129 set
zb = (zb.assign(Model=zb.model.map(MODEL_LABEL), Contrast=zb.pair.map(PAIR_LABEL),
                Categories=zb.n_categories.astype(str))
        .pivot_table(index=["Model", "Contrast", "Categories"], columns="domain_coef", values="cell", aggfunc="first")
        .rename(columns={"alone": "Domain coefficient, no control", "controlled": "Domain coefficient, co-occurrence controlled"}))
zl = lvis_diff[lvis_diff.metric.isin(["auc", "domain_coef_alone", "domain_coef_controlled"]) & lvis_diff.pair.str.endswith("lvis")]
zl = (zl.assign(cell=[fmt(r.diff, r.ci_lo, r.ci_hi, r.p_boot) for r in zl.itertuples()],
                Model=zl.model.map(MODEL_LABEL), Contrast=zl.pair.map(PAIR_LABEL), Categories=str(len(SETS["bv_vedb_things_lvis"][0])))
        .pivot_table(index=["Model", "Contrast", "Categories"], columns="metric", values="cell", aggfunc="first")
        .rename(columns={"auc": "AUC", "domain_coef_alone": "Domain coefficient, no control",
                         "domain_coef_controlled": "Domain coefficient, co-occurrence controlled"}))
tz = pd.concat([zb, zl]).sort_index(level=["Model"], sort_remaining=False)
tz.to_csv(CC_RESULTS / "supp_table_Z_cooccurrence_and_lvis.csv")

with open(CC_RESULTS / "supp_tables_YZ.md", "w") as f:
    f.write("**Supplementary Table Y.** Subject split (BabyView split by child, VEDB by session, THINGS by image): dataset differences in CDI-domain cluster strength on the "
            "split-averaged cross-half RDM (115 categories). Difference [95% category-bootstrap CI], two-sided p.\n\n")
    f.write(to_markdown(ty.reset_index()) + "\n\n")
    f.write("**Supplementary Table Z.** Same-domain coefficient (rank regression of category-pair distance on "
            "same-domain membership; larger = tighter clusters) without and with controls for frame co-occurrence and "
            "shared-context similarity (BabyView co-occurrence), and AUC against LVIS. Difference [95% CI], p.\n\n")
    f.write(to_markdown(tz.reset_index().fillna("—")) + "\n")
display(ty); display(tz.fillna("—"))

### Review checklist

- Does the BV crop count assert pass (co-occurrence built from the same crops as the embeddings)?
- Is every dataset controlled with the same covariates in the difference tests?
- Are bootstrap draws shared across datasets, with self-pairs dropped?
- Does the controlled same-domain coefficient remain significant for every dataset (domain structure is real)?
- Which dataset differences change sign or significance after control? (Interpretation lives in the README.)
- Co-occurrence also tracks meaning; the controlled coefficient is conservative. LVIS lacks a CLIP/size filter.